In [111]:
import Tensor as t
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml

We will stick to the row major order to align with numpy. This means our "dense" layers will be

$$\bold{Y} = \bold{X}\bold{W} + \bold{b}$$

Where $\bold{X}$ is the row vector in question. Might implement a technique called batching in the future.

In [112]:
# Fetch the MNIST dataset (this might take a minute to download)
mnist = fetch_openml('mnist_784', version=1, as_frame=False)

# Split into features (images) and labels
X, y = mnist["data"], mnist["target"]

print(f"Dataset shape: {X.shape}")

Dataset shape: (70000, 784)


In [113]:
def encode(val):
    z = np.zeros(10)
    z[int(val)] += 1
    return z

y_cleaned = np.array([encode(k) for k in y])

x_cleaned = X

In [114]:
print(x_cleaned.shape)

(70000, 784)


In [115]:
print(y_cleaned.shape)

(70000, 10)


In [ ]:
#current architecture: Dense(784 15) sAct softmax Dense(15 10) sAct softmax (done!)
#paramaters
w_1 = np.random.random_sample(size=(784, 15))
b_1 = np.random.random_sample(size=(1,15))

w_2 = np.random.random_sample(size=(15,10))
b_2 = np.random.random_sample(size=(1,10))

def pipeline(input):
    L_1 = (t.TensorNode(input,is_param=False).copy()) @ t.TensorNode(w_1) + t.TensorNode(b_1)
    L_1 = L_1.sAct()
    
    L_2 = L_1 @ t.TensorNode(w_2) + t.TensorNode(b_2)

    L_2 = L_2.sAct()
    return L_2


In [117]:
model = pipeline(x_cleaned[0])
modelCompiler = model.compile() 
#use model to actually get predictions and vector ouputs!
print(model.data)
print(y_cleaned[0])

[[ 80442.91697999  88110.51098753  95269.72261772 123581.87042704
   96995.83807447  88589.44831004  90362.05088711 129614.43094588
  104265.13864008 116098.79759837]]
[0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]


In [118]:
loss = (pipeline(x_cleaned) - t.TensorNode(y_cleaned,is_param=False)).norm_squared()

In [119]:
for j in range(10):
    lossTrainer = loss.compile()
    print(loss.data) # Mean Square error Currently

    lossTrainer.train()
    lossTrainer.update(1e-2)

    print(loss.data) # Mean Square Error Afterwards

7399765660586106.0
70000.0
70000.0
70000.0
70000.0
70000.0
70000.0
70000.0
70000.0
70000.0
70000.0
70000.0
70000.0
70000.0
70000.0
70000.0
70000.0
70000.0
70000.0
70000.0


WE DID IT. Below is the accuracy:

In [120]:
correct = 0
for j in range(50_000):
    modelCompiler.update_input(np.array([x_cleaned[j]]))
    if(str(np.argmax(model.data)) == y[j]):
        correct+=1

print("Raw correct: " + str(correct))
print("accruacy " + str(correct/50_000))

print("test dataset:")
correct = 0
for j in range(50_001,70_000):
    modelCompiler.update_input(x_cleaned[j])
    if(str(np.argmax(model.data)) == y[j]):
        correct+=1

print("Raw correct: " + str(correct))
print("accruacy " + str(correct/20_000))


AttributeError: 'numpy.ndarray' object has no attribute 'update_policy'

Check out data.npz to import paramaters